# OpenFold2/AlphaFold2 Ray Pipeline Example

This notebook demonstrates multi-stages ray pipeline for folding process.

## Imports

### Step 1: Setup Environment

In [2]:
%env ALPHAFOLD2_1_CKPT=/workspace/tensorrt_bionemo/examples/openfold2/af2_pt/alphafold2_1.pt
%env ENGINE_OUTPUT_DIR=/workspace/tensorrt_bionemo/examples/openfold2/alphafold2_1_engines

env: ALPHAFOLD2_1_CKPT=/workspace/tensorrt_bionemo/examples/openfold2/af2_pt/alphafold2_1.pt
env: ENGINE_OUTPUT_DIR=/workspace/tensorrt_bionemo/examples/openfold2/alphafold2_1_engines


In [3]:
import os
import time
from pathlib import Path
import torch
import ray

from tensorrt_bionemo.pipeline.processor.engine_proc import (
    EngineProcessorConfig, build_processor)
from tensorrt_bionemo.pipeline.stages.configs import (
    EngineStageConfig, FeatureGeneratorStageConfig, TokenizerStageConfig, ParserStageConfig, WriterStageConfig)
from tensorrt_bionemo.data.parsers import read_fasta
from tensorrt_bionemo.data.schemas import InputRequest, Polymer, MSARecord
SAMPLE_DIR = Path.cwd().parent / "data" / "samples" / "monomers"

# Set environment variables (uncomment and modify paths as needed)

# PyTorch backend only
ALPHAFOLD2_1_CKPT = os.environ.get("ALPHAFOLD2_1_CKPT", "alphafold2_1.pt")
# With TRT acceleration
ENGINE_OUTPUT_DIR = os.environ.get("ENGINE_OUTPUT_DIR", "alphafold2_1_engines")

MODEL_NAME = "alphafold2_1"
output_dir = "output"

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-24 11:33:54,985	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


[TensorRT-LLM][DEBUG] Registered plugin creator TriAttn version 1 in namespace tensorrt_bionemo
[TensorRT-LLM][DEBUG] Registered plugin creator IdentitySZ version 1 in namespace tensorrt_bionemo


2026-03-24 11:33:56,133	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


### Step 2: Build TensorRT Engines (if you want to try with TensorRT engine)

Build optimized TensorRT engines for the Evoformer module. This may take 10-30 minutes depending on your GPU.

In [4]:
# Build TensorRT engines (skipped if already built)
import subprocess
import shutil

subprocess.run('nvidia-smi', check=True)

# ============ TRT Build Configuration (single source of truth) ============
BUILD_CONFIG = {
    "model_name": MODEL_NAME,
    "module": "evoformer",
    "safetensor_dir": "evoformer_safetensors",
    "output_dir": ENGINE_OUTPUT_DIR,
    "dtype": "bfloat16",
    "max_seqlen": 1536,
    "min_seqlen": 16,
    "triangle_attn_backend": "CUEQUIV",
}

# Build commands derived from config
CONVERT_CMD = [
    "python", "convert_evoformer_checkpoint.py",
    "--model_name", BUILD_CONFIG["model_name"],
    "--output_dir", BUILD_CONFIG["safetensor_dir"],
    "--triangle_attn_backend", BUILD_CONFIG["triangle_attn_backend"],
]

BUILD_CMD = [
    "trtbnm-build",
    "--model", BUILD_CONFIG["model_name"],
    "--module", BUILD_CONFIG["module"],
    "--checkpoint_dir", BUILD_CONFIG["safetensor_dir"],
    "--max_seqlen", str(BUILD_CONFIG["max_seqlen"]),
    "--min_seqlen", str(BUILD_CONFIG["min_seqlen"]),
    "--output_dir", BUILD_CONFIG["output_dir"],
    "--weakly_dtype", BUILD_CONFIG["dtype"],
]

engine_file = os.path.join(BUILD_CONFIG["output_dir"], "rank0.engine")
if os.path.exists(engine_file):
    print(f"⏭️  TRT engines already exist at: {BUILD_CONFIG['output_dir']}")
    print("   Delete the directory and re-run this cell to rebuild.")
else:
    print(f"Building TRT engines (this takes 10-30 minutes)...")
    print(f"\nConfig: {BUILD_CONFIG}\n")
    
    # Clean up previous safetensor directory if exists
    if os.path.exists(BUILD_CONFIG["safetensor_dir"]):
        shutil.rmtree(BUILD_CONFIG["safetensor_dir"])
    
    print(f"Step 1: Converting checkpoint...")
    print(f"  Command: {' '.join(CONVERT_CMD)}\n")
    subprocess.run(CONVERT_CMD, check=True)

    print(f"\nStep 2: Building TRT engines...")
    print(f"  Command: {' '.join(BUILD_CMD)}\n")
    subprocess.run(BUILD_CMD, check=True)

    print(f"\n✅ TRT engines built successfully at: {BUILD_CONFIG['output_dir']}")

Tue Mar 24 11:34:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:41:00.0 Off |                    0 |
| N/A   37C    P0             45W /  300W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

2026-03-24 11:34:20,907 - tensorrt_bionemo.hubs.local - INFO - Loading alphafold2_1 from local filesystem /workspace/tensorrt_bionemo/examples/openfold2/af2_pt/alphafold2_1.pt


Total time of converting checkpoints: 00:00:02

Step 2: Building TRT engines...
  Command: trtbnm-build --model alphafold2_1 --module evoformer --checkpoint_dir evoformer_safetensors --max_seqlen 1536 --min_seqlen 16 --output_dir /workspace/tensorrt_bionemo/examples/openfold2/alphafold2_1_engines --weakly_dtype bfloat16

[TensorRT-LLM][DEBUG] Registered plugin creator TriAttn version 1 in namespace tensorrt_bionemo
[TensorRT-LLM][DEBUG] Registered plugin creator IdentitySZ version 1 in namespace tensorrt_bionemo
[03/24/2026-11:34:26] [TRT-LLM] [I] Building module evoformer for backend trt
[03/24/2026-11:34:26] [TRT-LLM] [W] Overriding # of builder profiles <= 2.
[03/24/2026-11:34:26] [TRT-LLM] [I] Module config dtype: float32, weakly_dtype: bfloat16
[03/24/2026-11:34:26] [TRT-LLM] [I] Building weakly-typed engine with dtype bfloat16.
[03/24/2026-11:34:27] [TRT-LLM] [I] Set dtype to float32.
[03/24/2026-11:34:27] [TRT] [I] [MemUsageChange] Init CUDA: CPU +0, GPU +0, now: CPU 153, GPU 42

---

## Helper functions

Once you have your model checkpoint and optionally built TRT engines, you can run the inference pipeline below.

In [4]:
def get_trt_accelerated_configs() -> dict:
    """Check if TRT engines are available from environment variables."""
    if os.path.exists(ENGINE_OUTPUT_DIR):
        return {"evoformer": {"checkpoint": ENGINE_OUTPUT_DIR, "backend": "trt"}}
    return None

def get_num_gpus() -> int:
    from tensorrt_bionemo.pipeline.processor.utils import get_available_gpu_count   
    return get_available_gpu_count()
    
def create_sample_requests(repeat: int = 50):
    """Create sample protein folding requests."""
    requests = []
    # sample_ids = ["T1031", "T1033", "T1047s1", "T1096"]
    sample_ids = ["T1047s1"]
    for i in range(repeat):
        for sample_id in sample_ids:
            sequence = read_fasta(str(SAMPLE_DIR / f"{sample_id}.fasta"))["sequences"][0]["sequence"]
            requests.append(
                InputRequest(
                    input_id=f"{sample_id}_{i}",
                    polymers=[Polymer(
                        chain_id="A",
                        sequence=sequence,
                        msas=[MSARecord(path=str(SAMPLE_DIR / "msas" / f"{sample_id}.a3m"))]
                    )]
                )
            )
    
    return requests
def run_pipeline(
    config: EngineProcessorConfig,
    trt_acc: dict = None, 
    repeat: int = 50
):
    """
    Run the protein structure prediction pipeline.
    
    Args:
        model_name: Model to use (e.g., "alphafold2_1")
        trt_acc: TensorRT acceleration config
        num_gpu_replicas: Number of GPU replicas for the folding engine stage
    """
    
    os.environ["RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION"] = "0.5"
    
    print(f"\n{'='*80}")
    print(f"OpenFold2/AlphaFold2 Structure Prediction Pipeline")
    print(f"{'='*80}")
    print(f"Model: {config.model_source}")
    
    print(f"Config: {config}")
    
    engine_kwargs = {}
    if trt_acc:
        print(f"Backend: PyTorch + Evoformer TensorRT")
        print(f"Accelerated modules:")
        engine_kwargs["accelerated_configs"] = trt_acc
        for module, path in trt_acc.items():
            print(f"  - {module}: {path}")
    else:
        print(f"Backend: PyTorch (Standard)")
    print(f"{'='*80}\n")
    
    processor = build_processor(config)
    
    requests = create_sample_requests(repeat)
    records = [{"record": req, "__record_id": req["input_id"]} for req in requests]
    
    print(f"Processing {len(records)} protein sequences...")
    
    ds = ray.data.from_items(records)
    
    ctx = ray.data.DataContext.get_current()
    ctx.max_errored_blocks = 100
    
    ds = processor(ds)
    
    success_count = 0
    error_count = 0
    results = []
    
    # materialize the dataset (batching, for the streaming mode don't push the line)
    start_time = time.time()
    ds = ds.materialize()
    elapsed_time = time.time() - start_time
    for row in ds.iter_rows():
        err = row.get("__inference_error__", {})
        if err.get("error_msg"):
            error_count += 1
            print(f"\n❌ Error in {row.get('__record_id', 'unknown')}:")
            print(f"   {err.get('error_msg')}")
        else:
            success_count += 1
            output_path = row.get("output_path")
            if output_path:
                results.append({
                    "id": row.get("__record_id"),
                    "path": output_path
                })
    
    
    print(f"\n{'='*80}")
    print(f"Pipeline Results")
    print(f"{'='*80}")
    print(f"Total time: {elapsed_time:.2f} seconds")
    print(f"Time per sequence: {elapsed_time/len(records):.2f} seconds")
    print(f"Successful: {success_count}/{len(records)}")
    print(f"Errors: {error_count}/{len(records)}")
    
    if results:
        print(f"\nOutput structures:")
        for result in results[:10]:
            print(f"  ✓ {result['id']}: {result['path']}")
        print(f"...")
    
    print(f"{'='*80}\n")
    
    return success_count, error_count, elapsed_time

## Configuration and Setup

Set up the model name and output directory. Make sure the required environment variables are set before running the pipeline.

## Run the Pipeline

Execute the pipeline with the configured settings.

In [5]:
import logging
import warnings
# First, get the handle for the logger you want to modify
ray_data_logger = logging.getLogger("ray.data")
ray_serve_logger = logging.getLogger("ray.serve")
ray_data_logger.setLevel(logging.ERROR)
ray_serve_logger.setLevel(logging.ERROR)
os.environ["TLLM_LOG_LEVEL"] = "ERROR"

warnings.filterwarnings("ignore", category=DeprecationWarning)

### 1. With single GPU

In [ ]:
from ray.data import DataContext
# Disable all progress bars
ray.shutdown()
DataContext.get_current().enable_progress_bars = False
print(f"Run on single GPU - {torch.cuda.get_device_name(0)}")
num_gpus = get_num_gpus()
print(f"Detected {num_gpus} GPU(s)")
trt_acc = get_trt_accelerated_configs() # Set to None to run only torch backend
try:
    config = EngineProcessorConfig(
        model_source=MODEL_NAME,
        executor_backend="ray",
        parser_stage=ParserStageConfig(compute=2),
        tokenizer_stage=TokenizerStageConfig(compute=2, num_cpus=2),
        feature_generator_stage=FeatureGeneratorStageConfig(compute=4, num_cpus=4),
        writer_stage=WriterStageConfig(
            compute=2,
            output_path=output_dir,
            format="pdb"
        ),
        engine_stage=EngineStageConfig(
            compute=1,
            num_cpus=4
        )
    )
    success, errors, elapsed = run_pipeline(config, trt_acc, repeat=400)
    
    if errors == 0:
        print(f"✅ Pipeline completed successfully!")
    else:
        print(f"⚠️  Pipeline completed with {errors} errors")
        
except Exception as e:
    print(f"\n❌ Pipeline failed with error:")
    print(f"   {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()

/usr/lib/python3.12/subprocess.py:1127: ResourceWarning: subprocess 3261507 is still running
  _warn("subprocess %s is still running" % self.pid,
/home/dev_user/.local/lib/python3.12/site-packages/ray/_private/node.py:1136: ResourceWarning: unclosed file <_io.TextIOWrapper name='/dev/null' mode='w' encoding='UTF-8'>
  process_info = ray._private.services.start_gcs_server(


Run on single GPU - NVIDIA H100 PCIe
Detected 4 GPU(s)

OpenFold2/AlphaFold2 Structure Prediction Pipeline
Model: alphafold2_1
Config: batch_size=1 accelerator_type=None concurrency=1 model_source='alphafold2_1' runtime_env=None max_pending_requests=None max_concurrent_batches=8 should_continue_on_error=False engine_kwargs={} parser_stage=ParserStageConfig(enabled=True, batch_size=None, compute=2, runtime_env=None, num_cpus=None, memory=None, compute_by_rows=True, drop_keys=None) tokenizer_stage=TokenizerStageConfig(enabled=True, batch_size=None, compute=2, runtime_env=None, num_cpus=2.0, memory=None, compute_by_rows=True, drop_keys=None) feature_generator_stage=FeatureGeneratorStageConfig(enabled=True, batch_size=None, compute=4, runtime_env=None, num_cpus=4.0, memory=None, compute_by_rows=True, drop_keys=None, init_context=None) writer_stage=WriterStageConfig(enabled=True, batch_size=None, compute=2, runtime_env=None, num_cpus=None, memory=None, compute_by_rows=True, drop_keys=None, 

/home/dev_user/.local/lib/python3.12/site-packages/ray/_private/utils.py:381: ResourceWarning: unclosed file <_io.TextIOWrapper name='/sys/fs/cgroup/cpu.max' mode='r' encoding='UTF-8'>
  max_file = open(cpu_max_file_name).read()
2026-02-24 09:24:22,046	INFO worker.py:1998 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


Processing 400 protein sequences...


(MapWorker(MapBatches(TokenizerUDF)) pid=3330622) /home/dev_user/.local/lib/python3.12/site-packages/ray/_common/utils.py:95: DeprecationWarning: There is no current event loop
(MapWorker(MapBatches(TokenizerUDF)) pid=3330622)   return asyncio.get_event_loop_policy().get_event_loop()
(pid=gcs_server) [2026-02-24 09:24:48,153 E 3329277 3329277] (gcs_server) gcs_server.cc:303: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(MapWorker(MapBatches(FeatureGeneratorUDF)) pid=3330625) /home/dev_user/.local/lib/python3.12/site-packages/ray/_common/utils.py:95: DeprecationWarning: There is no current event loop
(MapWorker(MapBatches(FeatureGeneratorUDF)) pid=3330625)   return asyncio.get_event_loop_policy().get_event_loop()
(MapWorker(MapBatches(FoldingEngineUDF)) pid=3330618) 2026-02-24 09:24:48,561 - tensorrt_bionemo.hubs.local - IN


Pipeline Results
Total time: 1030.92 seconds
Time per sequence: 2.58 seconds
Successful: 400/400
Errors: 0/400

Output structures:
  ✓ T1047s1_0: output/T1047s1_0.pdb
  ✓ T1047s1_1: output/T1047s1_1.pdb
  ✓ T1047s1_2: output/T1047s1_2.pdb
  ✓ T1047s1_3: output/T1047s1_3.pdb
  ✓ T1047s1_5: output/T1047s1_5.pdb
  ✓ T1047s1_4: output/T1047s1_4.pdb
  ✓ T1047s1_6: output/T1047s1_6.pdb
  ✓ T1047s1_7: output/T1047s1_7.pdb
  ✓ T1047s1_8: output/T1047s1_8.pdb
  ✓ T1047s1_9: output/T1047s1_9.pdb
...

✅ Pipeline completed successfully!


(pid=gcs_server) [2026-02-24 09:41:43,596 E 3329277 3329277] (gcs_server) gcs_actor_scheduler.cc:558: Failed to kill actor bd50b04cf3d70117d6a3228601000000, return status: Invalid: KillActor RPC failed for actor bd50b04cf3d70117d6a3228601000000: RpcError: RPC error: Socket closed rpc_code: 14


### 2. With multiple GPUs (Replica mode)

In [ ]:
from ray.data import DataContext
# Disable all progress bars
ray.shutdown()
DataContext.get_current().enable_progress_bars = False
print("Run with replica mode using all GPUs")
trt_acc = get_trt_accelerated_configs() # Set to None to run only torch backend
try:
    config = EngineProcessorConfig.create_default_replica_mode_config(MODEL_NAME, output_dir)
    success, errors, elapsed = run_pipeline(config, trt_acc, repeat=400)
    
    if errors == 0:
        print(f"✅ Pipeline completed successfully!")
    else:
        print(f"⚠️  Pipeline completed with {errors} errors")
        
except Exception as e:
    print(f"\n❌ Pipeline failed with error:")
    print(f"   {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()

Run with replica mode using all GPUs

OpenFold2/AlphaFold2 Structure Prediction Pipeline
Model: alphafold2_1
Config: batch_size=1 accelerator_type=None concurrency=1 model_source='alphafold2_1' runtime_env=None max_pending_requests=None max_concurrent_batches=8 should_continue_on_error=False engine_kwargs={} parser_stage=ParserStageConfig(enabled=True, batch_size=None, compute=4, runtime_env=None, num_cpus=None, memory=None, compute_by_rows=True, drop_keys=None) tokenizer_stage=TokenizerStageConfig(enabled=True, batch_size=None, compute=4, runtime_env=None, num_cpus=2.0, memory=None, compute_by_rows=True, drop_keys=None) feature_generator_stage=FeatureGeneratorStageConfig(enabled=True, batch_size=None, compute=4, runtime_env=None, num_cpus=4.0, memory=None, compute_by_rows=True, drop_keys=None, init_context=None) writer_stage=WriterStageConfig(enabled=True, batch_size=None, compute=4, runtime_env=None, num_cpus=None, memory=None, compute_by_rows=True, drop_keys=None, output_path='outpu

2026-02-24 09:18:32,239	INFO worker.py:1998 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
/home/dev_user/.local/lib/python3.12/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Processing 400 protein sequences...


(MapWorker(MapBatches(TokenizerUDF)) pid=3262396) /home/dev_user/.local/lib/python3.12/site-packages/ray/_common/utils.py:95: DeprecationWarning: There is no current event loop
(MapWorker(MapBatches(TokenizerUDF)) pid=3262396)   return asyncio.get_event_loop_policy().get_event_loop()
(pid=gcs_server) [2026-02-24 09:18:58,160 E 3261042 3261042] (gcs_server) gcs_server.cc:303: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(MapWorker(MapBatches(FeatureGeneratorUDF)) pid=3262416) /home/dev_user/.local/lib/python3.12/site-packages/ray/_common/utils.py:95: DeprecationWarning: There is no current event loop
(MapWorker(MapBatches(FeatureGeneratorUDF)) pid=3262416)   return asyncio.get_event_loop_policy().get_event_loop()
(MapWorker(MapBatches(FoldingEngineUDF)) pid=3262399) 2026-02-24 09:19:02,723 - tensorrt_bionemo.hubs.local - IN


Pipeline Results
Total time: 285.81 seconds
Time per sequence: 0.71 seconds
Successful: 400/400
Errors: 0/400

Output structures:
  ✓ T1047s1_0: output/T1047s1_0.pdb
  ✓ T1047s1_1: output/T1047s1_1.pdb
  ✓ T1047s1_2: output/T1047s1_2.pdb
  ✓ T1047s1_4: output/T1047s1_4.pdb
  ✓ T1047s1_3: output/T1047s1_3.pdb
  ✓ T1047s1_5: output/T1047s1_5.pdb
  ✓ T1047s1_6: output/T1047s1_6.pdb
  ✓ T1047s1_7: output/T1047s1_7.pdb
  ✓ T1047s1_8: output/T1047s1_8.pdb
  ✓ T1047s1_10: output/T1047s1_10.pdb
...

✅ Pipeline completed successfully!


(pid=gcs_server) [2026-02-24 09:23:30,588 E 3261042 3261042] (gcs_server) gcs_actor_scheduler.cc:558: Failed to kill actor 393c7ae642ef1b53e57bd0dd01000000, return status: Invalid: KillActor RPC failed for actor 393c7ae642ef1b53e57bd0dd01000000: RpcError: RPC error: Socket closed rpc_code: 14
(MapWorker(MapBatches(FeatureGeneratorUDF)) pid=3262438) /home/dev_user/.local/lib/python3.12/site-packages/ray/_common/utils.py:95: DeprecationWarning: There is no current event loop
(MapWorker(MapBatches(FeatureGeneratorUDF)) pid=3262438)   return asyncio.get_event_loop_policy().get_event_loop()
